# Fine-tune PhoBERT-base cho NER COVID-19 tiếng Việt — Hoàng Duyên

**Đồ án NLP · Đề 26 · PhoNER_COVID19** · Issue **T15** (fine-tune × 3 seed) · phục vụ BC1 mục (4), báo cáo mục 4.3–4.4 và 5.3

| | |
|---|---|
| Mô hình | `vinai/phobert-base` · đầu vào **mức từ (đã tách từ bằng RDRSegmenter)** |
| Đặc điểm cần nhớ | Pretrain riêng cho tiếng Việt (RoBERTa) trên ~20 GB văn bản **đã tách từ**, BPE 64K → đầu vào phải là bản `data/word`. |
| Chạy ở đâu | **Google Colab, GPU T4** (Runtime → Change runtime type → T4 GPU → Save) |
| Thời gian ước lượng | chạy thử 2 phút · Grid Search ~35–50 phút · Random Search ~35–50 phút · 3 seed ~45–90 phút |
| Kết quả cần nộp vào repo | `results/phobert_seed{42,123,2024}.json`, `results/du_doan/phobert_seed*.json`, `results/tim_kiem/phobert_*.json` |

**Quy trình (đúng chuẩn thực nghiệm):** tìm siêu tham số **chỉ trên tập dev** (Grid Search + Random Search, mục 4.3) → huấn luyện lại 3 seed
với cấu hình tốt nhất, chọn checkpoint theo F1 dev, **chấm test đúng 1 lần mỗi seed** → báo cáo mean ± std (std mẫu, n−1).
Cả 3 kiến trúc dùng **cùng script, cùng không gian tìm kiếm, cùng ngân sách** → so sánh công bằng (Dodge et al., EMNLP 2019).

> Có thể chạy từng phần qua nhiều buổi: mọi kết quả được sao lưu lên Google Drive sau mỗi lượt và tự khôi phục khi mở lại.

## 1. Kiểm tra GPU

In [ ]:
import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU → Save, rồi Runtime → Restart session"

## 2. Thư mục làm việc + Google Drive

In [ ]:
# Thư mục làm việc + Google Drive (Drive để SAO LƯU kết quả: Colab ngắt giữa chừng thì không mất)
import os, sys, json, shutil, glob, time
TREN_COLAB = "google.colab" in sys.modules
if TREN_COLAB:
    os.makedirs("/content/de26", exist_ok=True)
    os.chdir("/content/de26")
for d in ("src", "configs", "data", "results"):
    os.makedirs(d, exist_ok=True)
DRIVE = None
if TREN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE = "/content/drive/MyDrive/NLP_De26"
        os.makedirs(DRIVE, exist_ok=True)
        print("✅ Drive OK — kết quả được sao lưu ở", DRIVE)
    except Exception as loi:
        print("⚠️ Không gắn được Drive (vẫn chạy được, nhưng nhớ tải kết quả về trước khi tắt tab):", loi)
print("Thư mục làm việc:", os.getcwd())

## 3. Ghi code của nhóm (giống hệt thư mục `src/` trong repo — không cần token GitHub)
Chỉ cần bấm chạy; muốn đọc code thì mở từng ô.

In [ ]:
%%writefile configs/common.json
{
  "_ghi_chu": "Hợp đồng chung của nhóm. Cả 3 kiến trúc đọc chung file này để so sánh công bằng. Muốn đổi giá trị nào: mở PR + ghi lý do vào docs/QUY_UOC_CHUNG.md, không sửa riêng trong notebook.",
  "trang_thai": "ĐỀ XUẤT — chốt ở buổi họp 12/10/2026",

  "du_lieu": {
    "thu_muc": "data",
    "muc_tu": "data/word",
    "muc_am_tiet": "data/syllable",
    "ten_file": "{split}_{muc}.json",
    "dinh_dang": "Mỗi dòng 1 câu: {\"words\": [...], \"tags\": [...]}",
    "so_cau_mong_doi": {"train": 5027, "dev": 2000, "test": 3000}
  },

  "cac_loai_thuc_the": ["PATIENT_ID", "NAME", "AGE", "GENDER", "JOB", "LOCATION",
                         "ORGANIZATION", "SYMPTOM_AND_DISEASE", "TRANSPORTATION", "DATE"],
  "nhan": ["O",
           "B-PATIENT_ID", "I-PATIENT_ID", "B-NAME", "I-NAME", "B-AGE", "I-AGE",
           "B-GENDER", "I-GENDER", "B-JOB", "I-JOB", "B-LOCATION", "I-LOCATION",
           "B-ORGANIZATION", "I-ORGANIZATION", "B-SYMPTOM_AND_DISEASE", "I-SYMPTOM_AND_DISEASE",
           "B-TRANSPORTATION", "I-TRANSPORTATION", "B-DATE", "I-DATE"],

  "cham_diem": {
    "thu_vien": "seqeval",
    "mode": "strict",
    "scheme": "IOB2",
    "chi_so_chinh": "f1_micro",
    "do_lech_chuan": "std mẫu (chia n-1), numpy.std(..., ddof=1)",
    "chon_checkpoint": "f1_micro cao nhất trên dev; test chỉ dùng để báo cáo"
  },

  "seeds": [42, 123, 2024],

  "sieu_tham_so": {
    "learning_rate": 5e-5,
    "batch_size": 16,
    "gradient_accumulation_steps": 2,
    "so_epoch_toi_da": 15,
    "dung_som_sau": 3,
    "max_length": 256,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "fp16": true,
    "_ly_do": "Bài báo dùng lr 5e-5, batch 32, 30 epoch, dừng sớm 5. Nhóm giữ lr, batch hiệu dụng 16 x 2 = 32, nhưng giảm còn 15 epoch / dừng sớm 3 để kịp chạy 3 seed trước 18/10 trên Colab."
  },

  "mo_hinh": {
    "phobert":  {"ten": "vinai/phobert-base",            "muc_du_lieu": "muc_tu",      "nguoi": "Hoàng Duyên"},
    "xlmr":     {"ten": "xlm-roberta-base",              "muc_du_lieu": "muc_am_tiet", "nguoi": "Đức Tâm"},
    "mbert":    {"ten": "bert-base-multilingual-cased",  "muc_du_lieu": "muc_am_tiet", "nguoi": "Yến Trinh"},
    "phobert_khong_tach_tu": {"ten": "vinai/phobert-base", "muc_du_lieu": "muc_am_tiet", "nguoi": "Phúc", "ghi_chu": "Yêu cầu nâng cao (sau BC1)"}
  },

  "ten_file_ket_qua": "results/{khoa_mo_hinh}_seed{seed}.json  (vd: results/phobert_seed42.json, results/phobert_khong_tach_tu_seed42.json)"
}

In [ ]:
%%writefile src/nap_seqeval.py
"""nap_seqeval.py - Bảo đảm dùng được thư viện seqeval trên mọi máy (Colab, Kaggle, Windows, macOS).

Vì sao cần file này?
    seqeval 1.2.2 (bản mới nhất, 2020) chỉ phát hành dạng mã nguồn đóng gói kiểu cũ (setup.py), nên
    "pip install seqeval" bị lỗi "metadata-generation-failed" trên Python mới (Colab dùng Python 3.12+).
    seqeval viết bằng Python thuần, nên chỉ cần TẢI MÃ NGUỒN về rồi thêm vào sys.path là dùng được.

Cách dùng (đặt TRƯỚC mọi lệnh "from seqeval ..."):
    from nap_seqeval import dam_bao_seqeval
    dam_bao_seqeval()
"""

import os
import sys
import tarfile
import tempfile
import urllib.request

PHIEN_BAN = "1.2.2"
URL_PYPI = f"https://files.pythonhosted.org/packages/source/s/seqeval/seqeval-{PHIEN_BAN}.tar.gz"


def dam_bao_seqeval(thu_muc_cache=None):
    """Nạp seqeval. Nếu máy chưa có thì tải mã nguồn từ PyPI (không cần pip, không cần git).
    Đầu ra: đường dẫn tới thư mục chứa seqeval đang dùng."""
    try:
        import seqeval  # noqa: F401  (đã cài sẵn thì dùng luôn)
        return os.path.dirname(os.path.dirname(seqeval.__file__))
    except ModuleNotFoundError:
        pass

    thu_muc_cache = thu_muc_cache or os.path.join(tempfile.gettempdir(), "seqeval_src")
    thu_muc_ma_nguon = os.path.join(thu_muc_cache, f"seqeval-{PHIEN_BAN}")
    if not os.path.exists(os.path.join(thu_muc_ma_nguon, "seqeval", "__init__.py")):
        os.makedirs(thu_muc_cache, exist_ok=True)
        file_nen = os.path.join(thu_muc_cache, f"seqeval-{PHIEN_BAN}.tar.gz")
        urllib.request.urlretrieve(URL_PYPI, file_nen)
        with tarfile.open(file_nen, "r:gz") as tar:
            try:
                tar.extractall(thu_muc_cache, filter="data")   # Python mới: chỉ giải nén file thường, an toàn
            except TypeError:
                tar.extractall(thu_muc_cache)

    if thu_muc_ma_nguon not in sys.path:
        sys.path.insert(0, thu_muc_ma_nguon)
    import seqeval  # noqa: F401,E402
    return thu_muc_ma_nguon


if __name__ == "__main__":
    print("seqeval nạp từ:", dam_bao_seqeval())
    from seqeval.metrics import f1_score
    from seqeval.scheme import IOB2
    print("Thử chấm 1 câu:", f1_score([["B-AGE", "O"]], [["B-AGE", "O"]], mode="strict", scheme=IOB2))

In [ ]:
%%writefile src/data.py
"""data.py - Đọc dữ liệu PhoNER_COVID19 và tạo bảng nhãn dùng chung cho cả nhóm.

Người phụ trách: Đức Tâm · Issue T07
Cách tự kiểm tra (ở thư mục gốc repo):  python src/data.py
Xong khi in ra: train 5027 / dev 2000 / test 3000 câu cho CẢ 2 bản, 21 nhãn, 0 câu lệch số từ - số nhãn.

Tiền xử lý duy nhất: chuẩn hóa Unicode NFC cho từng token (nhãn giữ nguyên vì 1 token -> 1 token).
Lý do: mỗi phiên bản dữ liệu gốc có 1.052 token chưa ở dạng NFC (dấu viết tách rời chữ cái) lẫn với dạng dựng sẵn;
cùng một chữ "Hà" có thể là 2 chuỗi byte khác nhau -> tokenizer cắt khác nhau. NFC là chuẩn an toàn mặc định
cho văn bản tiếng Việt (bài giảng Session 2 - Words, Tokens; Unicode Standard Annex #15).
"""

import json
import os
import unicodedata
from collections import Counter

THU_MUC_GOC = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DUONG_DAN_CAU_HINH = os.path.join(THU_MUC_GOC, "configs", "common.json")
CAC_TAP = ["train", "dev", "test"]


def doc_cau_hinh(duong_dan=DUONG_DAN_CAU_HINH):
    """Đọc configs/common.json -> dict cấu hình chung của nhóm."""
    with open(duong_dan, encoding="utf-8") as f:
        return json.load(f)


def chuan_hoa_nfc(cac_tu):
    """Chuẩn hóa Unicode NFC cho danh sách token (không đổi số lượng token)."""
    return [unicodedata.normalize("NFC", tu) for tu in cac_tu]


def doc_file_json(duong_dan, nfc=True):
    """Đọc 1 file .json của PhoNER_COVID19 (mỗi dòng {"words": [...], "tags": [...]}).
    Đầu ra: (cac_cau, cac_nhan) - 2 list cùng độ dài; mỗi phần tử là list token / list nhãn của 1 câu."""
    cac_cau, cac_nhan = [], []
    with open(duong_dan, encoding="utf-8") as f:
        for dong in f:
            if not dong.strip():
                continue
            mau = json.loads(dong)
            cac_cau.append(chuan_hoa_nfc(mau["words"]) if nfc else mau["words"])
            cac_nhan.append(mau["tags"])
    return cac_cau, cac_nhan


def tai_du_lieu(muc="word", thu_muc_data=None, nfc=True):
    """Đọc đủ 3 tập của 1 phiên bản dữ liệu.
    muc = "word"     (đã tách từ bằng RDRSegmenter - dùng cho PhoBERT, CRF)
        = "syllable" (âm tiết - dùng cho XLM-R, mBERT, zero-shot, PhoBERT không tách từ)
    Đầu ra: {"train": (cac_cau, cac_nhan), "dev": (...), "test": (...)}"""
    if muc not in ("word", "syllable"):
        raise ValueError("muc phải là 'word' hoặc 'syllable'")
    thu_muc_data = thu_muc_data or os.path.join(THU_MUC_GOC, "data")
    du_lieu = {}
    for tap in CAC_TAP:
        duong_dan = os.path.join(thu_muc_data, muc, f"{tap}_{muc}.json")
        if not os.path.exists(duong_dan):
            raise FileNotFoundError(f"Không thấy {duong_dan}. Xem data/README.md để tải dữ liệu PhoNER_COVID19.")
        du_lieu[tap] = doc_file_json(duong_dan, nfc=nfc)
    return du_lieu


def tao_label2id(cau_hinh=None):
    """label2id, id2label CỐ ĐỊNH theo thứ tự khóa "nhan" trong configs/common.json (21 nhãn).
    Cả nhóm phải dùng đúng hàm này để id của nhãn giống nhau ở mọi mô hình."""
    cau_hinh = cau_hinh or doc_cau_hinh()
    label2id = {nhan: i for i, nhan in enumerate(cau_hinh["nhan"])}
    id2label = {i: nhan for nhan, i in label2id.items()}
    return label2id, id2label


def lay_thuc_the(nhan_cau):
    """Lấy danh sách thực thể (loai, vi_tri_dau, vi_tri_cuoi) từ dãy nhãn BIO (IOB2, vị trí tính cả 2 đầu).
    Viết tay (không cần seqeval) để file này chạy được ở mọi máy."""
    thuc_the, loai_hien_tai, dau = [], None, None
    for i, nhan in enumerate(list(nhan_cau) + ["O"]):
        bat_dau_moi = nhan.startswith("B-") or nhan == "O" or (nhan.startswith("I-") and nhan[2:] != loai_hien_tai)
        if bat_dau_moi and loai_hien_tai is not None:
            thuc_the.append((loai_hien_tai, dau, i - 1))
            loai_hien_tai = None
        if nhan.startswith("B-") or (nhan.startswith("I-") and loai_hien_tai is None):
            loai_hien_tai, dau = nhan[2:], i
    return thuc_the


def kiem_tra_du_lieu(du_lieu, label2id):
    """Kiểm tra từng tập: số token = số nhãn, nhãn nằm trong label2id, nhãn I-X đứng sai chỗ (vi phạm IOB2).
    Đầu ra: {tap: {"so_cau", "lech_do_dai", "nhan_la", "I_sai_cho", "so_thuc_the"}}"""
    ket_qua = {}
    for tap, (cac_cau, cac_nhan) in du_lieu.items():
        lech = sum(len(c) != len(n) for c, n in zip(cac_cau, cac_nhan))
        nhan_la = {x for n in cac_nhan for x in n if x not in label2id}
        i_sai = 0
        for n in cac_nhan:
            truoc = "O"
            for x in n:
                if x.startswith("I-") and truoc[2:] != x[2:]:
                    i_sai += 1
                truoc = x
        ket_qua[tap] = {"so_cau": len(cac_cau), "lech_do_dai": lech, "nhan_la": sorted(nhan_la),
                        "I_sai_cho": i_sai, "so_thuc_the": sum(len(lay_thuc_the(n)) for n in cac_nhan)}
    return ket_qua


def dem_thuc_the_theo_loai(cac_nhan):
    """Counter {loai: so_thuc_the} cho 1 tập."""
    return Counter(loai for n in cac_nhan for loai, _, _ in lay_thuc_the(n))


if __name__ == "__main__":
    cau_hinh = doc_cau_hinh()
    label2id, id2label = tao_label2id(cau_hinh)
    print("Số nhãn:", len(label2id), "(mong đợi 21)")
    for muc in ["word", "syllable"]:
        du_lieu = tai_du_lieu(muc)
        print(f"\n[{muc}] mong đợi {cau_hinh['du_lieu']['so_cau_mong_doi']}")
        for tap, kq in kiem_tra_du_lieu(du_lieu, label2id).items():
            print(f"  {tap:5s} {kq}")

In [ ]:
%%writefile src/train_ner.py
"""train_ner.py - Fine-tune 1 kiến trúc cho NER. DÙNG CHUNG cho PhoBERT / XLM-R / mBERT (chỉ đổi --mo_hinh).

Người phụ trách: Hoàng Duyên · Issue T10 (Phúc review)

Ý tưởng (giải thích ngắn cho báo cáo mục 4.4):
  1. Mỗi câu là danh sách token (từ hoặc âm tiết) kèm nhãn BIO.
  2. Tokenizer của mô hình cắt mỗi token thành 1..n mảnh subword. Nhãn thật đặt ở mảnh ĐẦU của mỗi token,
     các mảnh còn lại nhận -100 (bị bỏ qua khi tính loss) -> "căn nhãn với subword".
  3. AutoModelForTokenClassification = encoder pretrained + 1 lớp tuyến tính 768 -> 21 nhãn, softmax từng vị trí,
     loss cross-entropy trên các vị trí có nhãn (bài giảng Session 6: L = -Σ log p(y_t | x)).
  4. Huấn luyện bằng AdamW + warmup tuyến tính, chấm dev sau mỗi epoch bằng F1 mức thực thể (seqeval strict IOB2),
     giữ checkpoint dev tốt nhất, dừng sớm khi dev không tăng N epoch, rồi mới chấm test ĐÚNG 1 LẦN.

Cách chạy:
  python src/train_ner.py --mo_hinh phobert --seed 42                 # 1 lượt
  python src/train_ner.py --mo_hinh xlmr --tim_kiem random --so_lan 6  # tìm siêu tham số (mục 4.3)
  python src/train_ner.py --mo_hinh mbert --seed 42 --thu_nhanh 300    # chạy thử nhanh 300 câu, 1 epoch
"""

import argparse
import inspect
import json
import os
import sys
import time

THU_MUC_SRC = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, THU_MUC_SRC)

from nap_seqeval import dam_bao_seqeval  # noqa: E402

dam_bao_seqeval()

import numpy as np  # noqa: E402
import torch  # noqa: E402
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score  # noqa: E402
from seqeval.scheme import IOB2  # noqa: E402
from transformers import (AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification,  # noqa: E402
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

from data import THU_MUC_GOC, doc_cau_hinh, tai_du_lieu, tao_label2id  # noqa: E402

try:                                    # dùng hàm chấm điểm của Phúc nếu đã có trong repo (src/metrics.py)
    from metrics import tinh_metrics, luu_json  # noqa: E402
except ImportError:
    tinh_metrics = luu_json = None

CAC_LOAI = ["PATIENT_ID", "NAME", "AGE", "GENDER", "JOB", "LOCATION", "ORGANIZATION",
            "SYMPTOM_AND_DISEASE", "TRANSPORTATION", "DATE"]


# ----------------------------------------------------------------------------------------------------
# 1. Chấm điểm (giống hệt quy ước nhóm: seqeval, mode="strict", scheme=IOB2)
# ----------------------------------------------------------------------------------------------------
def cham_diem(ten, y_that, y_du_doan):
    """Trả về dict cùng định dạng với metrics.tinh_metrics() của Phúc."""
    if tinh_metrics is not None:
        return tinh_metrics(ten, y_that, y_du_doan)
    bao_cao = classification_report(y_that, y_du_doan, output_dict=True, zero_division=0, mode="strict", scheme=IOB2)
    return {
        "ten_mo_hinh": ten, "che_do_cham": "strict_IOB2",
        "so_thuc_the": int(sum(bao_cao[l]["support"] for l in CAC_LOAI if l in bao_cao)),
        "precision_micro": round(float(precision_score(y_that, y_du_doan, zero_division=0, mode="strict", scheme=IOB2)), 4),
        "recall_micro": round(float(recall_score(y_that, y_du_doan, zero_division=0, mode="strict", scheme=IOB2)), 4),
        "f1_micro": round(float(f1_score(y_that, y_du_doan, zero_division=0, mode="strict", scheme=IOB2)), 4),
        "f1_macro": round(float(f1_score(y_that, y_du_doan, average="macro", zero_division=0, mode="strict", scheme=IOB2)), 4),
        "f1_tung_loai": {l: round(float(bao_cao[l]["f1-score"]), 4) if l in bao_cao else 0.0 for l in CAC_LOAI},
    }


def ghi_json(du_lieu, duong_dan):
    os.makedirs(os.path.dirname(duong_dan) or ".", exist_ok=True)
    with open(duong_dan, "w", encoding="utf-8") as f:
        json.dump(du_lieu, f, ensure_ascii=False, indent=2)


# ----------------------------------------------------------------------------------------------------
# 2. Mã hóa câu + căn nhãn với subword (cùng 1 cách cho cả 3 kiến trúc -> so sánh công bằng)
# ----------------------------------------------------------------------------------------------------
class BoMaHoa:
    """Bọc tokenizer: mã hóa TỪNG token rồi ghép, có bộ nhớ đệm (nhanh hơn nhiều với tokenizer chậm của PhoBERT).
    Cách này chạy được cho mọi tokenizer, kể cả PhoBERT (không có word_ids())."""

    def __init__(self, tokenizer, max_length):
        self.tok = tokenizer
        self.max_length = max_length
        self.cache = {}

    def manh(self, tu):
        if tu not in self.cache:
            ids = self.tok.encode(tu, add_special_tokens=False)
            self.cache[tu] = ids if ids else [self.tok.unk_token_id]     # ký tự lạ -> <unk>
        return self.cache[tu]

    def ma_hoa(self, cac_tu, cac_nhan=None, label2id=None):
        """Đầu ra: dict input_ids, attention_mask, labels (nếu có nhãn), vi_tri_dau (vị trí mảnh đầu của từng token,
        -1 nếu token bị cắt vì vượt max_length)."""
        ids, nhan, vi_tri_dau = [self.tok.cls_token_id], [-100], []
        gioi_han = self.max_length - 1                                    # chừa chỗ cho </s> / [SEP]
        for k, tu in enumerate(cac_tu):
            manh = self.manh(tu)
            if len(ids) + len(manh) > gioi_han:                           # hết chỗ: các token còn lại bị cắt
                vi_tri_dau.extend([-1] * (len(cac_tu) - k))
                break
            vi_tri_dau.append(len(ids))
            ids.extend(manh)
            if cac_nhan is not None:
                nhan.extend([label2id[cac_nhan[k]]] + [-100] * (len(manh) - 1))
        ids.append(self.tok.sep_token_id)
        mau = {"input_ids": ids, "attention_mask": [1] * len(ids), "vi_tri_dau": vi_tri_dau}
        if cac_nhan is not None:
            nhan.append(-100)
            mau["labels"] = nhan
        return mau


class TapDuLieu(torch.utils.data.Dataset):
    def __init__(self, cac_mau):
        self.cac_mau = cac_mau

    def __len__(self):
        return len(self.cac_mau)

    def __getitem__(self, i):
        m = self.cac_mau[i]
        return {k: m[k] for k in ("input_ids", "attention_mask", "labels")}


def ma_hoa_tap(bo_ma_hoa, cac_cau, cac_nhan, label2id):
    cac_mau = [bo_ma_hoa.ma_hoa(c, n, label2id) for c, n in zip(cac_cau, cac_nhan)]
    so_cau_bi_cat = sum(1 for m in cac_mau if -1 in m["vi_tri_dau"])
    return cac_mau, so_cau_bi_cat


# ----------------------------------------------------------------------------------------------------
# 3. Dự đoán về lại mức token (để chấm trên ĐỦ nhãn thật, kể cả token bị cắt -> tính là "O")
# ----------------------------------------------------------------------------------------------------
@torch.no_grad()
def du_doan(model, cac_mau, id2label, collator, batch_size=64, tra_xac_suat=False):
    model.eval()
    thiet_bi = next(model.parameters()).device
    ket_qua, xac_suat = [], []
    for i in range(0, len(cac_mau), batch_size):
        lo = cac_mau[i:i + batch_size]
        batch = collator([{"input_ids": m["input_ids"], "attention_mask": m["attention_mask"]} for m in lo])
        batch = {k: v.to(thiet_bi) for k, v in batch.items()}
        with torch.autocast(device_type=thiet_bi.type, dtype=torch.float16, enabled=thiet_bi.type == "cuda"):
            logits = model(**batch).logits
        p = torch.softmax(logits.float(), dim=-1)
        nhan_id = p.argmax(-1).cpu().numpy()
        p_max = p.max(-1).values.cpu().numpy()
        for j, m in enumerate(lo):
            ket_qua.append([id2label[int(nhan_id[j, v])] if v >= 0 else "O" for v in m["vi_tri_dau"]])
            if tra_xac_suat:
                xac_suat.append([float(p_max[j, v]) if v >= 0 else 0.0 for v in m["vi_tri_dau"]])
    return (ket_qua, xac_suat) if tra_xac_suat else ket_qua


def sua_chuoi_bio(cac_nhan):
    """Hậu xử lý nhẹ: I-X đứng sau O hoặc sau loại khác -> đổi thành B-X (để thực thể không bị mất khi chấm strict).
    Đây là cách chuẩn thường dùng khi không có lớp CRF; ghi rõ trong báo cáo."""
    ra, truoc = [], "O"
    for n in cac_nhan:
        if n.startswith("I-") and truoc[2:] != n[2:]:
            n = "B-" + n[2:]
        ra.append(n)
        truoc = n
    return ra


# ----------------------------------------------------------------------------------------------------
# 4. Tham số huấn luyện (tương thích nhiều phiên bản transformers)
# ----------------------------------------------------------------------------------------------------
def tao_training_args(thu_muc_ra, sts, seed, so_epoch, co_danh_gia=True):
    tham_so = dict(
        output_dir=thu_muc_ra, learning_rate=sts["learning_rate"],
        per_device_train_batch_size=sts["batch_size"], per_device_eval_batch_size=64,
        gradient_accumulation_steps=sts.get("gradient_accumulation_steps", 1),
        num_train_epochs=so_epoch, weight_decay=sts.get("weight_decay", 0.0),
        warmup_ratio=sts.get("warmup_ratio", 0.0), lr_scheduler_type="linear",
        save_strategy="epoch" if co_danh_gia else "no", logging_steps=50,
        load_best_model_at_end=co_danh_gia, metric_for_best_model="f1_micro", greater_is_better=True,
        save_total_limit=1, fp16=torch.cuda.is_available() and sts.get("fp16", True),
        seed=seed, data_seed=seed, report_to="none", disable_tqdm=False,
    )
    chu_ky = inspect.signature(TrainingArguments.__init__).parameters
    tham_so["eval_strategy" if "eval_strategy" in chu_ky else "evaluation_strategy"] = "epoch"
    if "warmup_ratio" not in chu_ky:               # transformers >= 5: warmup_ratio bị bỏ, warmup_steps nhận số thực = tỉ lệ
        tham_so["warmup_steps"] = float(tham_so.pop("warmup_ratio"))
    if "save_only_model" in chu_ky:
        tham_so["save_only_model"] = True          # không lưu optimizer -> đỡ tốn ổ đĩa Colab
    return TrainingArguments(**{k: v for k, v in tham_so.items() if k in chu_ky or k in ("eval_strategy", "evaluation_strategy")})


def ham_cham_dev(id2label):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        y_that, y_doan = [], []
        for p_cau, l_cau in zip(pred, labels):
            mask = l_cau != -100
            y_that.append([id2label[int(x)] for x in l_cau[mask]])
            y_doan.append(sua_chuoi_bio([id2label[int(x)] for x in p_cau[mask]]))
        return {"f1_micro": f1_score(y_that, y_doan, mode="strict", scheme=IOB2, zero_division=0),
                "f1_macro": f1_score(y_that, y_doan, average="macro", mode="strict", scheme=IOB2, zero_division=0)}
    return compute_metrics


# ----------------------------------------------------------------------------------------------------
# 5. Một lượt huấn luyện đầy đủ (1 kiến trúc × 1 seed)
# ----------------------------------------------------------------------------------------------------
def huan_luyen(khoa_mo_hinh, seed, sieu_tham_so=None, so_epoch=None, dung_som=None, thu_nhanh=0,
               thu_muc_checkpoint=None, luu_mo_hinh=None, ghi_ket_qua=True, cham_test=True, nguoi_chay=None):
    """Huấn luyện 1 lượt và (mặc định) chấm test, ghi results/{khoa}_seed{seed}.json + results/du_doan/....
    sieu_tham_so: dict ghi đè configs/common.json (dùng khi đã tìm được bộ tốt hơn ở mục 4.3).
    Trả về dict kết quả."""
    cau_hinh = doc_cau_hinh()
    thong_tin = cau_hinh["mo_hinh"][khoa_mo_hinh]
    sts = dict(cau_hinh["sieu_tham_so"])
    sts.update(sieu_tham_so or {})
    so_epoch = so_epoch or sts["so_epoch_toi_da"]
    dung_som = sts["dung_som_sau"] if dung_som is None else dung_som
    muc = "word" if thong_tin["muc_du_lieu"] == "muc_tu" else "syllable"
    label2id, id2label = tao_label2id(cau_hinh)

    set_seed(seed)                                                    # 1) cố định seed TRƯỚC khi tạo model
    du_lieu = tai_du_lieu(muc)
    tok = AutoTokenizer.from_pretrained(thong_tin["ten"])
    bo_ma_hoa = BoMaHoa(tok, sts["max_length"])
    cau_train, nhan_train = du_lieu["train"]
    if thu_nhanh:
        cau_train, nhan_train, so_epoch = cau_train[:thu_nhanh], nhan_train[:thu_nhanh], 1
    mau_train, cat_train = ma_hoa_tap(bo_ma_hoa, cau_train, nhan_train, label2id)
    if thu_nhanh:                                                     # chạy thử: cắt cả dev cho nhanh
        du_lieu["dev"] = (du_lieu["dev"][0][:thu_nhanh], du_lieu["dev"][1][:thu_nhanh])
        du_lieu["test"] = (du_lieu["test"][0][:thu_nhanh], du_lieu["test"][1][:thu_nhanh])
    mau_dev, cat_dev = ma_hoa_tap(bo_ma_hoa, *du_lieu["dev"], label2id)
    print(f"[{khoa_mo_hinh} | seed {seed}] train {len(mau_train)} câu, dev {len(mau_dev)} câu; "
          f"câu bị cắt ở {sts['max_length']} subword: train {cat_train}, dev {cat_dev}")

    model = AutoModelForTokenClassification.from_pretrained(
        thong_tin["ten"], num_labels=len(label2id), id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
    collator = DataCollatorForTokenClassification(tok, pad_to_multiple_of=8 if torch.cuda.is_available() else None)
    thu_muc_ra = thu_muc_checkpoint or os.path.join("/tmp", "ckpt_ner", f"{khoa_mo_hinh}_seed{seed}")
    trainer = Trainer(
        model=model, args=tao_training_args(thu_muc_ra, sts, seed, so_epoch),
        train_dataset=TapDuLieu(mau_train), eval_dataset=TapDuLieu(mau_dev),
        data_collator=collator, compute_metrics=ham_cham_dev(id2label),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=dung_som)] if dung_som else [],
    )
    bat_dau = time.time()
    trainer.train()
    thoi_gian = round(time.time() - bat_dau)
    # Nạp lại checkpoint dev tốt nhất bằng from_pretrained (đổi tên khóa đúng cách). Lý do: checkpoint gốc của mBERT dùng tên
    # LayerNorm.gamma/beta; transformers 5 lưu lại đúng tên cũ nên bước load_best_model_at_end của Trainer bỏ sót các tham số
    # LayerNorm (cảnh báo "missing keys ... LayerNorm") -> mô hình "tốt nhất" bị lẫn LayerNorm của epoch cuối.
    tot_nhat_ckpt = trainer.state.best_model_checkpoint
    if tot_nhat_ckpt and os.path.isdir(tot_nhat_ckpt):
        trainer.model = AutoModelForTokenClassification.from_pretrained(tot_nhat_ckpt).to(trainer.args.device)

    # Epoch tốt nhất và F1 dev tốt nhất (đọc từ lịch sử huấn luyện)
    lich_su = [x for x in trainer.state.log_history if "eval_f1_micro" in x]
    tot_nhat = max(lich_su, key=lambda x: x["eval_f1_micro"]) if lich_su else {}

    # Chấm dev + test trên ĐỦ token (token bị cắt tính là "O")
    ket_qua_dev = cham_diem("dev", du_lieu["dev"][1], [sua_chuoi_bio(x) for x in du_doan(trainer.model, mau_dev, id2label, collator)])
    ket_qua = {"f1_micro_dev": ket_qua_dev["f1_micro"]}
    if cham_test:
        mau_test, cat_test = ma_hoa_tap(bo_ma_hoa, *du_lieu["test"], label2id)
        du_doan_test = [sua_chuoi_bio(x) for x in du_doan(trainer.model, mau_test, id2label, collator)]
        ket_qua = cham_diem(f"{khoa_mo_hinh} seed {seed}", du_lieu["test"][1], du_doan_test)
        ket_qua["so_cau_test_bi_cat"] = cat_test
    ket_qua.update({
        "khoa_mo_hinh": khoa_mo_hinh, "model": thong_tin["ten"], "muc_du_lieu": thong_tin["muc_du_lieu"],
        "seed": seed, "best_epoch": tot_nhat.get("epoch"), "so_epoch_da_chay": round(trainer.state.epoch or 0, 2),
        "f1_micro_dev": ket_qua_dev["f1_micro"], "f1_macro_dev": ket_qua_dev["f1_macro"],
        "lich_su_dev": [{"epoch": x["epoch"], "f1_micro": round(x["eval_f1_micro"], 4), "loss": round(x.get("eval_loss", 0), 4)}
                        for x in lich_su],
        "sieu_tham_so": {k: sts[k] for k in ("learning_rate", "batch_size", "gradient_accumulation_steps", "weight_decay",
                                             "warmup_ratio", "max_length", "fp16") if k in sts}
                        | {"so_epoch_toi_da": so_epoch, "dung_som_sau": dung_som},
        "so_cau_bi_cat": {"train": cat_train, "dev": cat_dev},
        "thoi_gian_train_giay": thoi_gian,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "moi_truong": ("Colab" if "google.colab" in sys.modules else "Kaggle" if os.path.exists("/kaggle") else "Máy cá nhân")
                      + " · " + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"),
        "thu_vien": {"torch": torch.__version__, "transformers": __import__("transformers").__version__,
                     "seqeval": getattr(__import__("seqeval"), "__version__", "1.2.2 (mã nguồn)")},
        "nguoi_chay": nguoi_chay or thong_tin.get("nguoi"),
        "thu_nhanh": bool(thu_nhanh),
    })

    if ghi_ket_qua and cham_test and not thu_nhanh:
        ghi_json(ket_qua, os.path.join(THU_MUC_GOC, "results", f"{khoa_mo_hinh}_seed{seed}.json"))
        ghi_json({"khoa_mo_hinh": khoa_mo_hinh, "seed": seed, "muc_du_lieu": muc, "tap": "test",
                  "du_doan": du_doan_test},
                 os.path.join(THU_MUC_GOC, "results", "du_doan", f"{khoa_mo_hinh}_seed{seed}.json"))
    if luu_mo_hinh:
        trainer.save_model(luu_mo_hinh)
        tok.save_pretrained(luu_mo_hinh)
        ghi_json({k: ket_qua.get(k) for k in ("khoa_mo_hinh", "model", "muc_du_lieu", "seed", "f1_micro", "f1_macro",
                                              "f1_micro_dev")}, os.path.join(luu_mo_hinh, "thong_tin_de26.json"))
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return ket_qua


# ----------------------------------------------------------------------------------------------------
# 6. Tìm siêu tham số: Grid Search / Random Search trên tập dev (báo cáo mục 4.3)
# ----------------------------------------------------------------------------------------------------
KHONG_GIAN_TIM_KIEM = {                          # cùng 1 không gian + cùng ngân sách cho cả 3 kiến trúc
    "learning_rate": [2e-5, 3e-5, 5e-5],
    "batch_size": [16, 32],
    "warmup_ratio": [0.0, 0.1],
    "weight_decay": [0.0, 0.01],
}
LUOI_NHO = {"learning_rate": [2e-5, 3e-5, 5e-5], "batch_size": [16, 32]}   # Grid Search: 3 × 2 = 6 cấu hình


def tim_sieu_tham_so(khoa_mo_hinh, kieu="random", so_lan=6, so_epoch=3, seed=42, thu_nhanh=0, nguoi_chay=None):
    """Grid Search (ParameterGrid) hoặc Random Search (ParameterSampler) của scikit-learn.
    Mỗi cấu hình: train `so_epoch` epoch trên train, chọn theo F1 micro trên DEV (không đụng test).
    Vì sao không dùng thẳng GridSearchCV? GridSearchCV cần estimator kiểu scikit-learn + k-fold cross-validation;
    với mô hình Transformer, k-fold tốn gấp k lần GPU và bộ dữ liệu đã có sẵn tập dev chuẩn, nên dùng đúng
    cơ chế duyệt lưới/lấy mẫu ngẫu nhiên của scikit-learn trên tập dev (hold-out validation)."""
    from sklearn.model_selection import ParameterGrid, ParameterSampler
    if kieu == "grid":
        cac_cau_hinh = list(ParameterGrid(LUOI_NHO))
    else:
        cac_cau_hinh = list(ParameterSampler(KHONG_GIAN_TIM_KIEM, n_iter=so_lan, random_state=seed))
    bang = []
    for i, ch in enumerate(cac_cau_hinh, 1):
        ch = {k: (float(v) if isinstance(v, (float, np.floating)) else int(v)) for k, v in ch.items()}
        print(f"\n===== {kieu} search {i}/{len(cac_cau_hinh)}: {ch} =====")
        ch_day_du = dict(ch, gradient_accumulation_steps=1)
        kq = huan_luyen(khoa_mo_hinh, seed, sieu_tham_so=ch_day_du, so_epoch=so_epoch, dung_som=0,
                        thu_nhanh=thu_nhanh, ghi_ket_qua=False, cham_test=False, nguoi_chay=nguoi_chay)
        bang.append({**ch, "f1_micro_dev": kq["f1_micro_dev"], "f1_macro_dev": kq["f1_macro_dev"],
                     "thoi_gian_giay": kq["thoi_gian_train_giay"]})
        print(f"   -> F1 micro dev = {kq['f1_micro_dev']:.4f}")
    tot = max(bang, key=lambda x: x["f1_micro_dev"])
    ket_qua = {"khoa_mo_hinh": khoa_mo_hinh, "kieu": kieu, "so_cau_hinh": len(bang), "so_epoch_moi_cau_hinh": so_epoch,
               "seed": seed, "khong_gian": LUOI_NHO if kieu == "grid" else KHONG_GIAN_TIM_KIEM,
               "bang": sorted(bang, key=lambda x: -x["f1_micro_dev"]), "tot_nhat": tot, "thu_nhanh": bool(thu_nhanh)}
    if not thu_nhanh:
        ghi_json(ket_qua, os.path.join(THU_MUC_GOC, "results", "tim_kiem", f"{khoa_mo_hinh}_{kieu}.json"))
    return ket_qua


def doc_cau_hinh_tot_nhat(khoa_mo_hinh, in_ra=True):
    """Bộ siêu tham số có F1 dev cao nhất qua CẢ Grid Search và Random Search đã chạy (results/tim_kiem/).
    Chưa chạy tìm kiếm thì trả {} (dùng giá trị mặc định trong configs/common.json)."""
    ung_vien = []
    for kieu in ("grid", "random"):
        p = os.path.join(THU_MUC_GOC, "results", "tim_kiem", f"{khoa_mo_hinh}_{kieu}.json")
        if os.path.exists(p):
            ung_vien.append((json.load(open(p, encoding="utf-8"))["tot_nhat"], kieu))
    if not ung_vien:
        return {}
    tot, kieu = max(ung_vien, key=lambda x: x[0]["f1_micro_dev"])
    if in_ra:
        print(f"Dùng cấu hình tốt nhất từ {kieu} search (F1 dev {tot['f1_micro_dev']:.4f}):",
              {k: tot[k] for k in KHONG_GIAN_TIM_KIEM if k in tot})
    return {k: tot[k] for k in KHONG_GIAN_TIM_KIEM if k in tot} | {"gradient_accumulation_steps": 1}


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="Fine-tune NER PhoNER_COVID19 (đề 26)")
    ap.add_argument("--mo_hinh", required=True, choices=["phobert", "xlmr", "mbert", "phobert_khong_tach_tu"])
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--tim_kiem", choices=["grid", "random"], help="chạy tìm siêu tham số thay vì huấn luyện")
    ap.add_argument("--so_lan", type=int, default=6, help="số cấu hình cho random search")
    ap.add_argument("--dung_cau_hinh_tot_nhat", action="store_true", help="dùng bộ tìm được ở results/tim_kiem/")
    ap.add_argument("--thu_nhanh", type=int, default=0)
    ap.add_argument("--luu_mo_hinh", default=None, help="thư mục lưu mô hình (cho demo)")
    a = ap.parse_args()
    if a.tim_kiem:
        print(json.dumps(tim_sieu_tham_so(a.mo_hinh, a.tim_kiem, a.so_lan, thu_nhanh=a.thu_nhanh)["tot_nhat"], indent=2))
    else:
        sts = doc_cau_hinh_tot_nhat(a.mo_hinh) if a.dung_cau_hinh_tot_nhat else None
        kq = huan_luyen(a.mo_hinh, a.seed, sieu_tham_so=sts, thu_nhanh=a.thu_nhanh, luu_mo_hinh=a.luu_mo_hinh)
        print(json.dumps({k: kq.get(k) for k in ("f1_micro", "f1_macro", "f1_micro_dev", "best_epoch")}, indent=2))

In [ ]:
%%writefile results/_MAU_ket_qua_finetune.json
{
  "_ghi_chu": "FILE MẪU — không phải kết quả thật. Tên file thật: results/{khoa_mo_hinh}_seed{seed}.json. 8 trường đầu do metrics.tinh_metrics() tạo; các trường sau do train_ner.py ghép thêm (cập nhật 17/10: thêm f1_macro_dev, lich_su_dev, so_cau_bi_cat, gpu; 'lr' đổi tên thành 'learning_rate').",
  "ten_mo_hinh": "xlmr seed 42",
  "che_do_cham": "strict_IOB2",
  "so_thuc_the": 11735,
  "precision_micro": null,
  "recall_micro": null,
  "f1_micro": null,
  "f1_macro": null,
  "f1_tung_loai": {
    "PATIENT_ID": null,
    "NAME": null,
    "AGE": null,
    "GENDER": null,
    "JOB": null,
    "LOCATION": null,
    "ORGANIZATION": null,
    "SYMPTOM_AND_DISEASE": null,
    "TRANSPORTATION": null,
    "DATE": null
  },
  "khoa_mo_hinh": "xlmr",
  "model": "xlm-roberta-base",
  "muc_du_lieu": "muc_am_tiet",
  "seed": 42,
  "best_epoch": null,
  "so_epoch_da_chay": null,
  "f1_micro_dev": null,
  "f1_macro_dev": null,
  "lich_su_dev": [
    {
      "epoch": 1.0,
      "f1_micro": null,
      "loss": null
    }
  ],
  "sieu_tham_so": {
    "learning_rate": 5e-05,
    "batch_size": 16,
    "gradient_accumulation_steps": 2,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_length": 256,
    "fp16": true,
    "so_epoch_toi_da": 15,
    "dung_som_sau": 3
  },
  "so_cau_bi_cat": {
    "train": null,
    "dev": null
  },
  "so_cau_test_bi_cat": null,
  "thoi_gian_train_giay": null,
  "gpu": "Tesla T4",
  "thu_vien": {
    "transformers": "x.y.z",
    "torch": "x.y.z",
    "seqeval": "x.y.z"
  },
  "moi_truong": "Colab T4 / Kaggle P100 / ...",
  "nguoi_chay": "Đức Tâm",
  "thu_nhanh": false
}

## 4. Tải dữ liệu PhoNER_COVID19 + kiểm tra

In [ ]:
# Dữ liệu công khai của VinAI: https://github.com/VinAIResearch/PhoNER_COVID19 (chỉ dùng cho nghiên cứu/học tập).
# Nếu ô này lỗi mạng: tải https://github.com/VinAIResearch/PhoNER_COVID19/archive/refs/heads/main.zip về máy,
# giải nén, rồi kéo 2 thư mục data/word và data/syllable vào thư mục /content/de26/data/ ở khung Files bên trái.
if not os.path.exists("data/word/train_word.json"):
    !git clone -q --depth 1 https://github.com/VinAIResearch/PhoNER_COVID19.git /tmp/PhoNER_COVID19
    !mkdir -p data && cp -r /tmp/PhoNER_COVID19/data/word /tmp/PhoNER_COVID19/data/syllable data/
!python src/data.py

## 5. Khôi phục kết quả cũ từ Drive (nếu đã chạy dở hôm trước) + hàm sao lưu

In [ ]:
KHOA = "phobert"
def sao_luu():
    """Chép results/ của mô hình này lên Drive (gọi sau mỗi lượt chạy)."""
    if DRIVE:
        for f in glob.glob(f"results/**/{KHOA}_*.json", recursive=True) + glob.glob(f"results/{KHOA}_*.json"):
            dich = os.path.join(DRIVE, "ket_qua_" + KHOA, os.path.relpath(f, "results"))
            os.makedirs(os.path.dirname(dich), exist_ok=True); shutil.copy(f, dich)
if DRIVE and os.path.isdir(os.path.join(DRIVE, "ket_qua_" + KHOA)):
    shutil.copytree(os.path.join(DRIVE, "ket_qua_" + KHOA), "results", dirs_exist_ok=True)
print("Đã có:", sorted(glob.glob(f"results/**/{KHOA}_*.json", recursive=True)) or "chưa có gì")

## 6. Chạy thử nhanh (300 câu, 1 epoch, ~2 phút)
Mục đích: bắt lỗi **trước** khi tốn hàng giờ GPU. F1 thấp là bình thường (mới học 300 câu). Ghi lại thời gian để ước lượng 1 epoch thật.

In [ ]:
sys.path.insert(0, "src")
import train_ner as T
bat_dau = time.time()
kq = T.huan_luyen(KHOA, 42, thu_nhanh=300)
print(f"\n✅ Chạy thử OK sau {time.time() - bat_dau:.0f} giây · F1 dev (300 câu) = {kq['f1_micro_dev']:.3f}")

## 6b. Kiểm tra 4 tiêu chí "xong" của Issue T10 (Duyên — chạy 1 lần, ~8 phút)
Tiêu chí ghi trong bảng phân công: (a) PhoBERT 1 epoch/500 câu không lỗi, số id = số nhãn ở mọi câu, F1 dev > 0;
(b) XLM-R và mBERT chạy ≥ 50 bước không lỗi; (c) 2 seed khác nhau cho loss khác nhau; (d) JSON đúng mẫu + checkpoint tốt nhất.
Chụp màn hình output 3 ô dưới, dán vào Pull Request của T10 làm bằng chứng.

In [ ]:
# (a) 1 epoch trên 500 câu + kiểm tra "số id = số nhãn" trên MỌI câu của cả 2 bản dữ liệu
kq_a = T.huan_luyen("phobert", 42, thu_nhanh=500, sieu_tham_so={"gradient_accumulation_steps": 1, "warmup_ratio": 0.0})
print("(a) F1 dev =", kq_a["f1_micro_dev"], "->", "ĐẠT" if kq_a["f1_micro_dev"] > 0 else "CHƯA ĐẠT")
label2id, _ = T.tao_label2id(T.doc_cau_hinh())
for khoa_kt, muc in [("phobert", "word"), ("xlmr", "syllable"), ("mbert", "syllable")]:
    tok = T.AutoTokenizer.from_pretrained(T.doc_cau_hinh()["mo_hinh"][khoa_kt]["ten"])
    bmh, loi, tong = T.BoMaHoa(tok, 256), 0, 0
    for tap, (cac_cau, cac_nhan) in T.tai_du_lieu(muc).items():
        for c, n in zip(cac_cau, cac_nhan):
            m = bmh.ma_hoa(c, n, label2id)
            so_nhan = sum(1 for x in m["labels"] if x != -100)            # vị trí có nhãn thật
            so_token = sum(1 for v in m["vi_tri_dau"] if v >= 0)          # token không bị cắt
            loi += len(m["input_ids"]) != len(m["labels"]) or so_nhan != so_token
            tong += 1
    print(f"(a) {khoa_kt}: {tong} câu, số câu lệch id/nhãn = {loi}")

In [ ]:
# (b) XLM-R và mBERT: 800 câu, batch 16, không tích lũy gradient -> 50 bước (thanh tiến trình hiện 50/50)
for khoa_kt in ["xlmr", "mbert"]:
    T.huan_luyen(khoa_kt, 42, thu_nhanh=800, sieu_tham_so={"gradient_accumulation_steps": 1})
    print(f"(b) {khoa_kt}: chạy xong 50 bước không lỗi")

In [ ]:
# (c) 2 seed khác nhau -> loss khác nhau (chứng minh seed có tác dụng, nên mean ± std mới có nghĩa)
k42 = T.huan_luyen("phobert", 42, thu_nhanh=300)
k123 = T.huan_luyen("phobert", 123, thu_nhanh=300)
print("(c) loss dev seed 42 =", k42["lich_su_dev"][0]["loss"], "| seed 123 =", k123["lich_su_dev"][0]["loss"])
# (d) kiểm tra sau ô 8 (lượt chạy thật mới ghi JSON): so khóa với file mẫu
def kiem_tra_d(seed=42):
    mau = json.load(open("results/_MAU_ket_qua_finetune.json", encoding="utf-8")) if os.path.exists("results/_MAU_ket_qua_finetune.json") else None
    kq = json.load(open(f"results/phobert_seed{seed}.json", encoding="utf-8"))
    if mau:
        print("(d) thiếu khóa so với mẫu:", (set(mau) - set(kq) - {"_ghi_chu"}) or "không")
    print("(d) checkpoint tốt nhất (thư mục mô hình):", os.listdir(f"/content/mo_hinh_{seed}") if os.path.isdir(f"/content/mo_hinh_{seed}") else "chưa có")

## 7. Tìm siêu tham số — báo cáo mục 4.3

**Grid Search** duyệt **mọi** tổ hợp của 1 lưới nhỏ; **Random Search** lấy ngẫu nhiên một số tổ hợp trong không gian rộng hơn
(Bergstra & Bengio, JMLR 2012: với cùng ngân sách, random search thường tìm được cấu hình tốt hơn vì không phí lượt vào tham số ít ảnh hưởng).
Dùng đúng `ParameterGrid` / `ParameterSampler` của scikit-learn — cũng là hai bộ sinh cấu hình bên trong `GridSearchCV` / `RandomizedSearchCV`.

*Vì sao không gọi thẳng `GridSearchCV`?* `GridSearchCV` đòi estimator kiểu scikit-learn và **k-fold cross-validation** → với Transformer
tốn gấp k lần GPU; PhoNER_COVID19 lại có sẵn tập **dev chuẩn** → dùng hold-out validation trên dev (thực hành chuẩn khi fine-tune).
Test **không** được dùng ở bước này.

| Cách | Không gian | Số cấu hình | Mỗi cấu hình |
|---|---|---|---|
| Grid Search | learning rate ∈ {2e-5, 3e-5, 5e-5} × batch ∈ {16, 32} | 6 | 3 epoch, seed 42 |
| Random Search | thêm warmup ∈ {0, 0.1}, weight decay ∈ {0, 0.01} (24 tổ hợp) | 6 (lấy ngẫu nhiên, random_state = 42) | 3 epoch, seed 42 |

Hết thời gian GPU? Đổi `SO_EPOCH_TIM_KIEM = 2` (ghi rõ trong báo cáo) — **cả 3 kiến trúc phải dùng cùng giá trị**.

In [ ]:
SO_EPOCH_TIM_KIEM = 3
import pandas as pd
for kieu in ["grid", "random"]:
    if os.path.exists(f"results/tim_kiem/{KHOA}_{kieu}.json"):
        print(f"(đã có {kieu} search — bỏ qua; xóa file trong results/tim_kiem/ nếu muốn chạy lại)")
        continue
    T.tim_sieu_tham_so(KHOA, kieu=kieu, so_lan=6, so_epoch=SO_EPOCH_TIM_KIEM, seed=42, nguoi_chay="Hoàng Duyên")
    sao_luu()
for kieu in ["grid", "random"]:
    r = json.load(open(f"results/tim_kiem/{KHOA}_{kieu}.json", encoding="utf-8"))
    print(f"\n===== {kieu.upper()} SEARCH ({r['so_cau_hinh']} cấu hình × {r['so_epoch_moi_cau_hinh']} epoch) =====")
    display(pd.DataFrame(r["bang"]))
CAU_HINH_TOT = T.doc_cau_hinh_tot_nhat(KHOA)

## 8. Huấn luyện chính thức 3 seed với cấu hình tốt nhất
Mỗi seed: tối đa 15 epoch, dừng sớm sau 3 epoch dev không tăng, giữ checkpoint dev tốt nhất, chấm test 1 lần.
Mô hình của từng seed lưu tạm ở `/content/mo_hinh_<seed>` để lát nữa giữ lại seed có **F1 dev** cao nhất cho demo
(chọn theo dev, không theo test — tránh "nhìn trộm" test).

In [ ]:
for seed in [42, 123, 2024]:
    if os.path.exists(f"results/{KHOA}_seed{seed}.json"):
        print(f"seed {seed}: đã có kết quả — bỏ qua"); continue
    kq = T.huan_luyen(KHOA, seed, sieu_tham_so=CAU_HINH_TOT, luu_mo_hinh=f"/content/mo_hinh_{seed}", nguoi_chay="Hoàng Duyên")
    sao_luu()
    print(f"\n>>> seed {seed}: F1 micro test = {kq['f1_micro']:.4f} · macro = {kq['f1_macro']:.4f} · "
          f"dev = {kq['f1_micro_dev']:.4f} · epoch tốt nhất = {kq['best_epoch']} · {kq['thoi_gian_train_giay'] / 60:.1f} phút\n")

## 9. Bảng kết quả 3 seed (mean ± std) — chép vào báo cáo mục 4.4 / 5.3

In [ ]:
import numpy as np
kq3 = [json.load(open(f"results/{KHOA}_seed{s}.json", encoding="utf-8")) for s in [42, 123, 2024]]
bang = pd.DataFrame([{"Seed": k["seed"], "Micro-F1 test": k["f1_micro"], "Macro-F1 test": k["f1_macro"],
                      "Precision": k["precision_micro"], "Recall": k["recall_micro"], "Micro-F1 dev": k["f1_micro_dev"],
                      "Epoch tốt nhất": k["best_epoch"], "Phút": round(k["thoi_gian_train_giay"] / 60, 1)} for k in kq3])
display(bang)
for cot in ["Micro-F1 test", "Macro-F1 test"]:
    print(f"{cot}: {bang[cot].mean():.4f} ± {bang[cot].std(ddof=1):.4f}  (std mẫu, n−1)")
print("\nF1 từng loại (mean 3 seed):")
display(pd.DataFrame([k["f1_tung_loai"] for k in kq3]).mean().round(4).to_frame("F1"))
print("Môi trường:", kq3[0]["gpu"], kq3[0]["thu_vien"], "· siêu tham số:", kq3[0]["sieu_tham_so"])

## 10. Tải kết quả về máy → đưa vào repo bằng GitHub Desktop
Ô dưới tạo `ket_qua_phobert.zip` (chỉ file JSON nhỏ, **không** có dữ liệu gốc). Giải nén vào thư mục repo trên máy
(`NLP-phoner-covid19-26/`, chọn *ghi đè*), rồi commit theo `docs/HUONG_DAN_3_BC2.md`.

In [ ]:
import zipfile
with zipfile.ZipFile(f"ket_qua_{KHOA}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(set(glob.glob(f"results/**/{KHOA}_*.json", recursive=True))):
        z.write(f); print("  +", f)
if TREN_COLAB:
    from google.colab import files
    files.download(f"ket_qua_{KHOA}.zip")

## 11. Lưu mô hình tốt nhất cho demo Streamlit (seed có F1 dev cao nhất)
Mô hình nặng (~540 MB) → **không** đưa lên GitHub. Ô dưới nén và chép lên Drive
(`NLP_De26/mo_hinh_phobert.zip`); Yến Trinh tải về, giải nén vào `app/mo_hinh/phobert/` để chạy demo.

In [ ]:
tot = max(kq3, key=lambda k: k["f1_micro_dev"])
print("Giữ mô hình seed", tot["seed"], "(F1 dev", tot["f1_micro_dev"], ")")
thu_muc = f"/content/mo_hinh_{tot['seed']}"
if os.path.isdir(thu_muc):
    shutil.make_archive(f"/content/mo_hinh_{KHOA}", "zip", thu_muc)
    if DRIVE:
        shutil.copy(f"/content/mo_hinh_{KHOA}.zip", DRIVE); print("✅ Đã chép lên Drive:", f"{DRIVE}/mo_hinh_{KHOA}.zip")
    elif TREN_COLAB:
        from google.colab import files; files.download(f"/content/mo_hinh_{KHOA}.zip")
    else:
        print("Đã nén:", f"/content/mo_hinh_{KHOA}.zip")
else:
    print("Không thấy thư mục mô hình (phiên Colab đã bị reset sau khi train). Chạy lại seed", tot["seed"],
          "bằng: T.huan_luyen(KHOA, seed, sieu_tham_so=CAU_HINH_TOT, luu_mo_hinh=..., ghi_ket_qua=False)")

## 12. (Tùy chọn) Đẩy mô hình lên Hugging Face Hub — chỉ làm nếu mô hình này tốt nhất nhóm (Issue T28)
Cần token **Write** của Hugging Face lưu trong Colab Secrets (biểu tượng 🔑 bên trái) với tên `HF_TOKEN`. Repo để **private**.

In [ ]:
DAY_LEN_HUB = False          # đổi thành True khi nhóm đã thống nhất đây là mô hình tốt nhất
TEN_REPO_HUB = "ten-tai-khoan-hf/de26-phoner-phobert"
if DAY_LEN_HUB:
    from google.colab import userdata
    from transformers import AutoModelForTokenClassification, AutoTokenizer
    tk = userdata.get("HF_TOKEN")
    m = AutoModelForTokenClassification.from_pretrained(thu_muc); t = AutoTokenizer.from_pretrained(thu_muc)
    m.push_to_hub(TEN_REPO_HUB, private=True, token=tk); t.push_to_hub(TEN_REPO_HUB, private=True, token=tk)
    from huggingface_hub import upload_file
    upload_file(path_or_fileobj=f"{thu_muc}/thong_tin_de26.json", path_in_repo="thong_tin_de26.json",
                repo_id=TEN_REPO_HUB, token=tk)
    print("✅ https://huggingface.co/" + TEN_REPO_HUB)

## 13. Ghi chú cho báo cáo (điền sau khi chạy)
- **Môi trường** (mục 4.6 / BC1 mục 4): GPU, phiên bản torch/transformers (in ở ô 9), thời gian mỗi seed.
- **Siêu tham số chọn được** và bảng Grid/Random Search (mục 4.3).
- **Khó khăn gặp phải + cách xử lý** (cô yêu cầu): ví dụ hết GPU Colab, mất kết nối (đã có sao lưu Drive), câu bị cắt ở 256 subword
  (xem số `so_cau_bi_cat` trong file JSON), seqeval không cài được bằng pip (nạp từ mã nguồn — `src/nap_seqeval.py`).
- **Issue cần đóng:** T15 (khi đủ 3 file JSON trong repo) · viết báo cáo: T19.